# Qubit LRB-D with sdim 1.4.0 DEM: SI1000 and SD6, one- and two-ancilla folded code

This notebook plots the four production LRB-D runs of the qubit `[[5,1,2]]` folded code, the qubit counterpart of `notebooks/04-qutrit-one-and-relay-two-ancilla-si1000-sd6.ipynb`. Circuits, check layouts, noise models, grids and fit settings are the qutrit ones with the local dimension set to 2:

| | SI1000 | SD6 |
|---|---|---|
| one ancilla per stabilizer (`folded_qubit`) | ✓ | ✓ |
| two ancillas for the boundary stabilizers (`folded_qubit_split_unif`) | ✓ | ✓ |

Shared with the qutrit notebook:

- **Simulator.** Circuits are sampled with sdim 1.4.0's exact compact detector error model (`sdim.dem.DetectorErrorModel`), cross-checked against sdim's Pauli-frame sampler in `tests/test_dem_backend.py`.
- **Clifford group.** Random Cliffords are drawn uniformly from the 24-element single-qubit Clifford group (4 Paulis × 6 symplectic words). `tests/test_folded_qubit_code.py` checks that every encoded Clifford and its inverse act as the intended logical gate in both layouts.
- **Noise models.** SI1000 and SD6 follow Gidney, Newman, Fowler and Broughton (2021), generalized to qudits, on an as-soon-as-possible moment schedule with idle noise. Ancillas are reset right before each check round.
- **Axis.** Every plot uses the swept parameter $p$ itself, which in both models is the two-qudit gate depolarizing probability, instead of the earlier $\lambda = 5p$.
- **RB comparison.** Physical single-qubit RB runs under the same noise model at the same $p$ (`results/RB`), so LRB and RB share the $p$ axis without a separate comparison grid.


### 0. Environment check

The kernel should be the repo venv (`.venv-sdim140`), which pins sdim 1.4.0.

In [1]:
from importlib.metadata import version
version("sdim")

'1.4.0'

### 1. Shared imports and configuration

Same fit, threshold, GIF and paper settings as the earlier DEM notebooks, except the axis is the two-qudit gate error $p$ (`p_axis_scale=1.0`).

In [2]:
from pathlib import Path
import sys
import importlib

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "lrb" / "lrb_plotting.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing src/lrb/lrb_plotting.py")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import lrb.lrb_plotting as _lrb_plotting
import lrb.lrb_gif_plotting as _lrb_gif_plotting

# Ensure notebook uses latest local edits to plotting/GIF modules.
importlib.reload(_lrb_plotting)
importlib.reload(_lrb_gif_plotting)

from lrb.lrb_plotting import (
    LRBPlotFitConfig,
    LRBResultsPlotter,
    LRBThresholdConfig,
    LRBPaperPlotConfig,
)
from lrb.lrb_gif_plotting import LRBGifAnimator, LRBGifRenderConfig


def resolve_working_folder(
    code_name: str,
    lrb_root=None,
) -> str:
    root = Path(lrb_root) if lrb_root is not None else REPO_ROOT / "LRB-experiment-data-slurm"
    code_marker = root / f"working-folder-{code_name}.txt"
    legacy_marker = root / "working-folder.txt"

    def marker_path(marker: Path):
        if not marker.exists():
            return None
        run_name = marker.read_text(encoding="utf-8").strip()
        if not run_name:
            return None
        run_path = root / run_name
        if not run_path.exists():
            return None
        return str(run_path) + "/"

    for marker in (code_marker, legacy_marker):
        maybe_path = marker_path(marker)
        if maybe_path is not None:
            return maybe_path

    candidates = sorted(
        root.glob(f"Run-*-{code_name}*"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        return str(candidates[0]) + "/"

    raise FileNotFoundError(
        f"No run folder found for code '{code_name}' under {root.resolve()}"
    )


fit_config = LRBPlotFitConfig(
    a_fixed=0.0,
    y_ceiling=1.0,
    min_fit_points_lrb=2,
    min_fit_points_rb=3,
    f_min=1e-8,
    f_max=1.0000000,
    rb_a_fixed=0.0,
    # SI1000 and SD6 are parameterized by the two-qudit gate depolarizing
    # probability p (Gidney et al. 2021); plot against it directly.
    p_axis_scale=1.0,
    p_axis_label="Two-qudit gate error rate $p$",
    p_axis_symbol="p",
    p_axis_column=None,
)

threshold_config = LRBThresholdConfig(
    tol=5e-4,
    require_consecutive=2,
    ignore_first_n=3,
    err_floor=2e-3,
    tail_min_prefix=6,
    tail_drop_tol_abs=0.01,
    tail_drop_tol_rel=0.25,
    p_min=None,
    p_max=None,
    min_lrb_n_points_keep=0,
    min_rb_n_points_keep=0,
    zoom_half_window_points=4,
    zoom_pad_frac=0.25,
    min_zoom_span=0.01,
    p_window_before_points=8,
    p_window_after_points=5,
    min_points_in_view=6,
    view_pad_frac=0.15,
    bootstrap_reps_error=500,
    bootstrap_reps_threshold=1000,
    bootstrap_ci_level=0.68,
    bootstrap_use_sem=True,
    bootstrap_seed=12345,
    paper_mode=True,
    paper_cols=2,
    paper_panel_width_in=3.2,
    paper_panel_height_in=2.5,
)

# Shared check ranges for all multi-check plotting/table methods.
const_check_min = 0
const_check_max = 0
unif_check_min = 1
unif_check_max = 22

# Optional pseudo-threshold fit controls.
pseudo_do_fit = False
pseudo_fit_model = "exp"  # or "poly"
pseudo_fit_degree = 1


gif_render_config = LRBGifRenderConfig(
    fps=10,
    dpi=120,
    hold_frames=1,
    transition_frames=6,
)

paper_plot_config = LRBPaperPlotConfig(
    enabled=True,
    column_layout="double",
    rows_per_page=3,
    pairs_per_row=2,
    num_prob_samples=6,
    # Circuit-level range on the production grid:
    # p = 1e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2.
    prob_sample_indices=(1, 3, 4, 5, 6, 8),
    row_height_in=1.6,
    panel_title_fontsize=7.0,
    axis_label_fontsize=6.6,
    tick_fontsize=6.0,
    legend_fontsize=5.7,
    line_width=0.85,
    marker_size=2.6,
    errorbar_capsize=1.2,
    show_legend_first_row_only=False,
    compact_fit_legend=True,
    legend_on_rejected_panel=True,
)

def show_error_rate_uncertainty_table(table_csv: str, label: str) -> pd.DataFrame:
    frame = pd.read_csv(table_csv)
    coordinate_columns = [
        c for c in ("lrb_lambda", "rb_p", "lambda", "p_raw", "p")
        if c in frame.columns
    ]
    sort_column = "lrb_lambda" if "lrb_lambda" in frame.columns else coordinate_columns[0]
    needed = [
        "check_num",
        *coordinate_columns,
        "lrb_error_rate",
        "lrb_error_rate_ci_low",
        "lrb_error_rate_ci_high",
        "rb_error_rate",
        "rb_error_rate_ci_low",
        "rb_error_rate_ci_high",
    ]
    missing = [c for c in needed if c not in frame.columns]
    if missing:
        raise ValueError(
            f"Missing uncertainty columns in {table_csv}: {missing}. "
            "Re-run the table generation cell."
        )

    view = frame[needed].copy()
    view["lrb_unc_minus"] = view["lrb_error_rate"] - view["lrb_error_rate_ci_low"]
    view["lrb_unc_plus"] = view["lrb_error_rate_ci_high"] - view["lrb_error_rate"]
    view["rb_unc_minus"] = view["rb_error_rate"] - view["rb_error_rate_ci_low"]
    view["rb_unc_plus"] = view["rb_error_rate_ci_high"] - view["rb_error_rate"]
    view = view.sort_values(["check_num", sort_column]).reset_index(drop=True)

    print(f"[{label}] error-rate uncertainty table")
    display(view)
    return view

def show_pseudo_threshold_uncertainty_table(
    summary_csv: str,
    label: str,
    check_min=None,
    check_max=None,
) -> pd.DataFrame:
    frame = pd.read_csv(summary_csv)
    threshold_column = (
        "threshold_lrb_lambda"
        if "threshold_lrb_lambda" in frame.columns
        else "threshold_lambda"
        if "threshold_lambda" in frame.columns
        else "threshold_p"
    )
    threshold_std_column = f"{threshold_column}_std_boot"
    threshold_ci_low_column = f"{threshold_column}_ci_low"
    threshold_ci_high_column = f"{threshold_column}_ci_high"
    needed = ["check_num", threshold_column]
    missing = [c for c in needed if c not in frame.columns]
    if missing:
        raise ValueError(
            f"Missing pseudo-threshold columns in {summary_csv}: {missing}. "
            "Re-run threshold generation first."
        )

    view = frame.copy()
    if check_min is not None:
        view = view[view["check_num"] >= int(check_min)]
    if check_max is not None:
        view = view[view["check_num"] <= int(check_max)]
    view = view[view[threshold_column].notna()].copy()

    keep = ["check_num", threshold_column]
    has_std = threshold_std_column in view.columns
    has_ci = (
        threshold_ci_low_column in view.columns
        and threshold_ci_high_column in view.columns
    )
    if has_std:
        keep.append(threshold_std_column)
    if has_ci:
        keep.extend([threshold_ci_low_column, threshold_ci_high_column])

    table = view[keep].copy()
    if has_ci:
        table["threshold_unc_minus"] = (
            table[threshold_column] - table[threshold_ci_low_column]
        )
        table["threshold_unc_plus"] = (
            table[threshold_ci_high_column] - table[threshold_column]
        )
    elif has_std:
        table["threshold_unc_minus"] = table[threshold_std_column]
        table["threshold_unc_plus"] = table[threshold_std_column]
    else:
        raise ValueError(
            f"No pseudo-threshold uncertainty columns found in {summary_csv}."
        )

    table = table.sort_values(["check_num"]).reset_index(drop=True)
    print(f"[{label}] pseudo-threshold uncertainty table")
    display(table)
    return table

### 2. Run selection and output audit

Runs are named explicitly so the notebook does not depend on the mutable `working-folder-*.txt` pointers. The audit requires every LRB and RB CSV before plotting.

In [3]:
RUNS_ROOT = REPO_ROOT / "LRB-experiment-data-slurm"
RUN_NAMES = {
    "single_si1000": "Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140",
    "single_sd6": "Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140",
    "split_si1000": "Run-2026-10-31-10-12-58-folded_qubit_split_unif-split-ancilla-si1000-sdim140",
    "split_sd6": "Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140",
}
EXPECTED_METADATA = {
    "single_si1000": ("folded_qubit", "si1000"),
    "single_sd6": ("folded_qubit", "sd6"),
    "split_si1000": ("folded_qubit_split_unif", "si1000"),
    "split_sd6": ("folded_qubit_split_unif", "sd6"),
}


def read_int_list(path: Path) -> list[int]:
    return [int(v) for v in path.read_text().replace(",", " ").split()]


def read_float_list(path: Path) -> list[float]:
    return [float(v) for v in path.read_text().replace(",", " ").split()]


def audit_run(label: str, run_path: Path) -> pd.DataFrame:
    code_name = (run_path / "code_name.txt").read_text().strip()
    noise_model = (run_path / "noise_model.txt").read_text().strip()
    if (code_name, noise_model) != EXPECTED_METADATA[label]:
        raise ValueError(f"{label}: found {(code_name, noise_model)}, expected {EXPECTED_METADATA[label]}")
    probs = read_float_list(run_path / "probs.txt")
    const_checks = read_int_list(run_path / "check_const.txt")
    unif_checks = read_int_list(run_path / "check_unif.txt")
    rows = []
    for idx, prob in enumerate(probs):
        lrb_path = run_path / "results" / "LRB" / str(idx)
        done_path = run_path / "progress" / str(idx) / "done.txt"
        rows.append({
            "run": label,
            "p_index": idx,
            "p": prob,
            "done_flag": done_path.read_text().strip() if done_path.exists() else "missing",
            "const_csvs": len(list((lrb_path / "const_check_data").glob("*.csv"))),
            "expected_const_csvs": len(const_checks),
            "unif_csvs": len(list((lrb_path / "unif_check_data").glob("*.csv"))),
            "expected_unif_csvs": len(unif_checks),
            "rb_csv_present": (run_path / "results" / "RB" / f"{idx}.csv").exists(),
        })
    return pd.DataFrame(rows)


run_paths = {label: RUNS_ROOT / name for label, name in RUN_NAMES.items()}
audit = pd.concat([audit_run(label, path) for label, path in run_paths.items()], ignore_index=True)
summary = audit.groupby("run").agg(
    indices=("p_index", "count"),
    done=("done_flag", lambda s: int((s == "1").sum())),
    lrb_csvs=("const_csvs", "sum"),
    unif_csvs=("unif_csvs", "sum"),
    expected_const=("expected_const_csvs", "sum"),
    expected_unif=("expected_unif_csvs", "sum"),
    rb_csvs=("rb_csv_present", "sum"),
)
display(summary)
incomplete = audit[
    (audit["done_flag"] != "1")
    | (audit["const_csvs"] != audit["expected_const_csvs"])
    | (audit["unif_csvs"] != audit["expected_unif_csvs"])
    | (~audit["rb_csv_present"])
]
if not incomplete.empty:
    raise ValueError("Incomplete runs/indices:\n" + incomplete.to_string())

,indices,done,lrb_csvs,unif_csvs,expected_const,expected_unif,rb_csvs
run,,,,,,,
single_sd6,15,15,345,330,345,330,15
single_si1000,15,15,345,330,345,330,15
split_sd6,15,15,345,330,345,330,15
split_si1000,15,15,345,330,345,330,15


### 3. Build plotters

One `LRBResultsPlotter` per run. With `p_axis_scale=1.0` and no `rb_comparison_probs.txt`, each plotter pairs LRB at $p$ with physical RB at the same $p$ from `results/RB`.

In [4]:
plotters = {
    label: LRBResultsPlotter(working_folder=str(path) + "/", fit_config=fit_config)
    for label, path in run_paths.items()
}
for label, plotter in plotters.items():
    rb_source, rb_probabilities = plotter._rb_table_source()
    print(f"{label}: {plotter.code_title}")
    print(f"  p grid: {plotter.probabilities}")
    print(f"  RB source: {rb_source}")
    if max(abs(a - b) for a, b in zip(plotter.probabilities, rb_probabilities)) > 0:
        raise ValueError(f"{label}: RB grid is not the LRB p grid.")

single_si1000: $[[5,1,2]]_2$, SI1000
  p grid: [0.0, 0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02, 0.03, 0.04, 0.05]
  RB source: /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/RB
single_sd6: $[[5,1,2]]_2$, SD6
  p grid: [0.0, 0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02, 0.03, 0.04, 0.05]
  RB source: /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140/results/RB
split_si1000: $[[5,1,2]]_2$ split-ancilla, SI1000
  p grid: [0.0, 0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02, 0.03, 0.04, 0.05]
  RB source: /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slu

### 4. Constant-check plots

`const_check=0` (the direct terminal-X check) for every run, as in the earlier notebooks.

In [5]:
const_outputs = {}
for label, plotter in plotters.items():
    print(f"=== {label}")
    const_outputs[label] = plotter.plot_all_const_checks(
        show=True,
        check_min=0,
        check_max=0,
        paper_config=paper_plot_config,
    )
const_outputs

=== single_si1000


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/const-0-Summary-Graph-NoFit.pdf
=== single_sd6


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140/results/plots/const-0-Summary-Graph-NoFit.pdf
=== split_si1000


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-58-folded_qubit_split_unif-split-ancilla-si1000-sdim140/results/plots/const-0-Summary-Graph-NoFit.pdf
=== split_sd6


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/const-0-Summary-Graph-NoFit.pdf


{'single_si1000': ['/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/const-0-Summary-Graph-NoFit.pdf'],
 'single_sd6': ['/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140/results/plots/const-0-Summary-Graph-NoFit.pdf'],
 'split_si1000': ['/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-58-folded_qubit_split_unif-split-ancilla-si1000-sdim140/results/plots/const-0-Summary-Graph-NoFit.pdf'],
 'split_sd6': ['/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/const-0-Summary-G

### 5. Uniform-check plots

Interval checks `1..4` for every run; raise `unif_plot_max` to `unif_check_max` for all intervals.

In [6]:
unif_plot_min, unif_plot_max = 1, 4
unif_outputs = {}
for label, plotter in plotters.items():
    print(f"=== {label}")
    unif_outputs[label] = plotter.plot_all_unif_checks(
        show=True,
        check_min=unif_plot_min,
        check_max=unif_plot_max,
        paper_config=paper_plot_config,
    )
unif_outputs

=== single_si1000


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-1-Summary-Graph-Fit.pdf


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-2-Summary-Graph-Fit.pdf


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-3-Summary-Graph-Fit.pdf


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-4-Summary-Graph-Fit.pdf
=== single_sd6


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140/results/plots/unif-1-Summary-Graph-Fit.pdf


... (8 outputs left out)

[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/unif-2-Summary-Graph-Fit.pdf


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/unif-3-Summary-Graph-Fit.pdf


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/unif-4-Summary-Graph-Fit.pdf


{'single_si1000': ['/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-1-Summary-Graph-Fit.pdf',
  '/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-2-Summary-Graph-Fit.pdf',
  '/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-3-Summary-Graph-Fit.pdf',
  '/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-4-Summary-Graph-Fit.pdf'],
 'single_sd6': ['/pscratch/sd/s/sohan100/worktrees/LRB

### 6. LRB-vs-RB tables and threshold summaries

LRB error rates and physical RB error rates are both placed at the same two-qudit gate error $p$. Uncertainties are bootstrap intervals as before.

In [7]:
point_ci_columns = [
    "lrb_error_rate_std_boot",
    "lrb_error_rate_ci_low",
    "lrb_error_rate_ci_high",
    "rb_error_rate_std_boot",
    "rb_error_rate_ci_low",
    "rb_error_rate_ci_high",
]
force_rebuild_uncertainty_table = False


def table_has_point_uncertainty(path: Path) -> bool:
    if not path.exists():
        return False
    columns = pd.read_csv(path, nrows=0).columns
    return all(col in columns for col in point_ci_columns)


table_csvs, summary_csvs, uncertainty_tables, pseudo_threshold_tables = {}, {}, {}, {}
for label, plotter in plotters.items():
    print(f"=== {label}")
    table_csv_path = Path(plotter.out_dir) / "unif_lrb_vs_rb_table_all_mixed_fits.csv"
    if force_rebuild_uncertainty_table or not table_has_point_uncertainty(table_csv_path):
        table_csvs[label] = plotter.build_unif_lrb_vs_rb_table_mixed_fits(
            write_per_check_tables=True,
            check_min=unif_check_min,
            check_max=unif_check_max,
            bootstrap_reps=threshold_config.bootstrap_reps_error,
            bootstrap_ci_level=threshold_config.bootstrap_ci_level,
            bootstrap_use_sem=threshold_config.bootstrap_use_sem,
            bootstrap_seed=threshold_config.bootstrap_seed,
        )
    else:
        table_csvs[label] = str(table_csv_path)
    summary_csvs[label] = plotter.plot_all_unif_threshold_graphs(
        threshold_config=threshold_config,
        table_csv_path=table_csvs[label],
        show=True,
        check_min=unif_check_min,
        check_max=4,
    )
    uncertainty_tables[label] = show_error_rate_uncertainty_table(table_csvs[label], label=label)
    pseudo_threshold_tables[label] = show_pseudo_threshold_uncertainty_table(
        summary_csvs[label],
        label=label,
        check_min=unif_check_min,
        check_max=unif_check_max,
    )

=== single_si1000


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-threshold-error-vs-p-paper-grid.pdf
[THRESHOLD] Uniform Interval Check = 1: p_LRB = p_RB = 0.01 +- 0.000209
[THRESHOLD] Uniform Interval Check = 2: p_LRB = p_RB = 0.0075 +- 1.31e-05
[THRESHOLD] Uniform Interval Check = 3: p_LRB = p_RB = 0.00502143 +- 0.00102
[THRESHOLD] Uniform Interval Check = 4: p_LRB = p_RB = 0.00315162 +- 2.96e-05
[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif_thresholds_summary_monotone_trim_zoom_pwindow.csv
[single_si1000] error-rate uncertainty table


,check_num,lrb_lambda,rb_p,p_raw,p,lrb_error_rate,lrb_error_rate_ci_low,lrb_error_rate_ci_high,rb_error_rate,rb_error_rate_ci_low,rb_error_rate_ci_high,lrb_unc_minus,lrb_unc_plus,rb_unc_minus,rb_unc_plus
0,1,0.0000,0.0000,0.0000,0.0000,0.000000e+00,0.000000e+00,2.500000e-08,0.000000,0.000000,0.000000,0.000000e+00,2.500000e-08,0.000000e+00,0.000000e+00
1,1,0.0001,0.0001,0.0001,0.0001,0.000000e+00,0.000000e+00,7.500000e-08,0.000033,0.000032,0.000034,0.000000e+00,7.500000e-08,9.524554e-07,9.495026e-07
2,1,0.0002,0.0002,0.0002,0.0002,4.292500e-07,2.261250e-07,6.597500e-07,0.000070,0.000068,0.000072,2.031250e-07,2.305000e-07,1.689975e-06,1.397777e-06
3,1,0.0005,0.0005,0.0005,0.0005,4.755585e-06,4.301211e-06,5.189632e-06,0.000172,0.000169,0.000175,4.543737e-07,4.340475e-07,2.922112e-06,2.880952e-06
4,1,0.0010,0.0010,0.0010,0.0010,2.041223e-05,1.965303e-05,2.110977e-05,0.000339,0.000334,0.000343,7.592071e-07,6.975403e-07,4.244641e-06,4.064903e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,22,0.0150,0.0150,0.0150,0.0150,1.069728e-01,1.067318e-01,1.072341e-01,0.005098,0.005051,0.005143,2.409920e-04,2.612640e-04,4.713408e-05,4.553161e-05
326,22,0.0200,0.0200,0.0200,0.0200,1.442448e-01,1.437209e-01,1.447872e-01,0.006757,0.006689,0.006818,5.238560e-04,5.423840e-04,6.858124e-05,6.138988e-05
327,22,0.0300,0.0300,0.0300,0.0300,2.296936e-01,2.282579e-01,2.310649e-01,0.010155,0.010065,0.010238,1.435648e-03,1.371264e-03,9.048260e-05,8.326068e-05
328,22,0.0400,0.0400,0.0400,0.0400,2.947688e-01,2.929738e-01,2.969479e-01,0.013472,0.013351,0.013585,1.794992e-03,2.179104e-03,1.209260e-04,1.129386e-04


[single_si1000] pseudo-threshold uncertainty table


,check_num,threshold_lrb_lambda,threshold_lrb_lambda_std_boot,threshold_lrb_lambda_ci_low,threshold_lrb_lambda_ci_high,threshold_unc_minus,threshold_unc_plus
0,1,0.010000,0.000209,0.010000,0.010000,0.00000,0.000000
1,2,0.007500,0.000013,0.007500,0.007500,0.00000,0.000000
2,3,0.005021,0.001017,0.005011,0.007500,0.00001,0.002479
3,4,0.003152,0.000030,0.003122,0.003181,0.00003,0.000030


=== single_sd6


... (10 outputs left out)

[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/unif-threshold-error-vs-p-paper-grid.pdf
[THRESHOLD] Uniform Interval Check = 1: p_LRB = p_RB = 0.0174099 +- 0.000449
[THRESHOLD] Uniform Interval Check = 2: p_LRB = p_RB = 0.0125 +- 0.000198
[THRESHOLD] Uniform Interval Check = 3: p_LRB = p_RB = 0.00781453 +- 7.16e-05
[THRESHOLD] Uniform Interval Check = 4: p_LRB = p_RB = 0.00580013 +- 4.84e-05
[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/unif_thresholds_summary_monotone_trim_zoom_pwindow.csv
[split_sd6] error-rate uncertainty table


,check_num,lrb_lambda,rb_p,p_raw,p,lrb_error_rate,lrb_error_rate_ci_low,lrb_error_rate_ci_high,rb_error_rate,rb_error_rate_ci_low,rb_error_rate_ci_high,lrb_unc_minus,lrb_unc_plus,rb_unc_minus,rb_unc_plus
0,1,0.0000,0.0000,0.0000,0.0000,0.000000,0.000000,2.500000e-08,0.000000,0.000000,0.000000,0.000000e+00,2.500000e-08,0.000000,0.000000
1,1,0.0001,0.0001,0.0001,0.0001,0.000001,0.000001,1.630361e-06,0.000349,0.000345,0.000353,2.426625e-07,2.911987e-07,0.000004,0.000004
2,1,0.0002,0.0002,0.0002,0.0002,0.000005,0.000004,5.217534e-06,0.000691,0.000683,0.000699,3.522375e-07,4.066512e-07,0.000009,0.000007
3,1,0.0005,0.0005,0.0005,0.0005,0.000032,0.000031,3.329139e-05,0.001719,0.001701,0.001735,1.003735e-06,9.095087e-07,0.000017,0.000017
4,1,0.0010,0.0010,0.0010,0.0010,0.000126,0.000123,1.285081e-04,0.003458,0.003423,0.003491,2.601095e-06,2.630167e-06,0.000035,0.000033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,22,0.0150,0.0150,0.0150,0.0150,0.219930,0.217991,2.220249e-01,0.049381,0.048952,0.049811,1.939328e-03,2.094944e-03,0.000429,0.000430
326,22,0.0200,0.0200,0.0200,0.0200,0.280301,0.277563,2.828553e-01,0.064505,0.063916,0.065014,2.738272e-03,2.554528e-03,0.000589,0.000509
327,22,0.0300,0.0300,0.0300,0.0300,0.352765,0.348182,3.571939e-01,0.093331,0.092299,0.094261,4.583600e-03,4.428752e-03,0.001032,0.000929
328,22,0.0400,0.0400,0.0400,0.0400,0.421514,0.416833,4.260408e-01,0.115426,0.113921,0.116901,4.681488e-03,4.526832e-03,0.001504,0.001475


[split_sd6] pseudo-threshold uncertainty table


,check_num,threshold_lrb_lambda,threshold_lrb_lambda_std_boot,threshold_lrb_lambda_ci_low,threshold_lrb_lambda_ci_high,threshold_unc_minus,threshold_unc_plus
0,1,0.017410,0.000449,0.017055,0.017856,0.000355,0.000446
1,2,0.012500,0.000198,0.012260,0.012659,0.000240,0.000159
2,3,0.007815,0.000072,0.007742,0.007884,0.000072,0.000069
3,4,0.005800,0.000048,0.005755,0.005849,0.000045,0.000049


### 7. Pseudo-threshold trend vs interval check

The focused `1..4` pseudo-threshold trend for every run.

In [8]:
pseudo_pdfs = {}
for label, plotter in plotters.items():
    pseudo_pdfs[label] = plotter.plot_unif_pseudo_thresholds_vs_interval_check(
        check_min=unif_check_min,
        check_max=4,
        do_fit=pseudo_do_fit,
        fit_model=pseudo_fit_model,
        fit_degree=pseudo_fit_degree,
        paper_mode=paper_plot_config.enabled,
        show=True,
    )
pseudo_pdfs

[PSEUDO] Uniform Interval Check = 1: p_LRB = p_RB = 0.01 +- 0.000209
[PSEUDO] Uniform Interval Check = 2: p_LRB = p_RB = 0.0075 +- 1.31e-05
[PSEUDO] Uniform Interval Check = 3: p_LRB = p_RB = 0.00502143 +- 0.00102
[PSEUDO] Uniform Interval Check = 4: p_LRB = p_RB = 0.00315162 +- 2.96e-05
[FIT] fit disabled by do_fit=False.


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-1-to-4-pseudo-threshold-vs-interval-check-fit.pdf
[PSEUDO] Uniform Interval Check = 1: p_LRB = p_RB = 0.0208506 +- 0.000303
[PSEUDO] Uniform Interval Check = 2: p_LRB = p_RB = 0.0151226 +- 0.000184
[PSEUDO] Uniform Interval Check = 3: p_LRB = p_RB = 0.00954191 +- 0.000122
[PSEUDO] Uniform Interval Check = 4: p_LRB = p_RB = 0.00765424 +- 6.98e-05
[FIT] fit disabled by do_fit=False.


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140/results/plots/unif-1-to-4-pseudo-threshold-vs-interval-check-fit.pdf
[PSEUDO] Uniform Interval Check = 1: p_LRB = p_RB = 0.01 +- 9.47e-05
[PSEUDO] Uniform Interval Check = 2: p_LRB = p_RB = 0.0064586 +- 8.57e-05
[PSEUDO] Uniform Interval Check = 3: p_LRB = p_RB = 0.00355048 +- 4.13e-05
[PSEUDO] Uniform Interval Check = 4: p_LRB = p_RB = 0.0026067 +- 1.44e-05
[FIT] fit disabled by do_fit=False.


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-58-folded_qubit_split_unif-split-ancilla-si1000-sdim140/results/plots/unif-1-to-4-pseudo-threshold-vs-interval-check-fit.pdf
[PSEUDO] Uniform Interval Check = 1: p_LRB = p_RB = 0.0174099 +- 0.000449
[PSEUDO] Uniform Interval Check = 2: p_LRB = p_RB = 0.0125 +- 0.000198
[PSEUDO] Uniform Interval Check = 3: p_LRB = p_RB = 0.00781453 +- 7.16e-05
[PSEUDO] Uniform Interval Check = 4: p_LRB = p_RB = 0.00580013 +- 4.84e-05
[FIT] fit disabled by do_fit=False.


[OK] wrote /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/unif-1-to-4-pseudo-threshold-vs-interval-check-fit.pdf


{'single_si1000': '/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/unif-1-to-4-pseudo-threshold-vs-interval-check-fit.pdf',
 'single_sd6': '/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140/results/plots/unif-1-to-4-pseudo-threshold-vs-interval-check-fit.pdf',
 'split_si1000': '/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-58-folded_qubit_split_unif-split-ancilla-si1000-sdim140/results/plots/unif-1-to-4-pseudo-threshold-vs-interval-check-fit.pdf',
 'split_sd6': '/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_

### 8. Four-configuration comparison

Logical error per Clifford with a check after every Clifford (uniform interval 1) for all four runs, against physical single-qubit RB under each noise model, versus the two-qudit gate error $p$. Points without a valid fit are omitted.

In [9]:
import matplotlib.pyplot as plt
import numpy as np

comparison_dir = RUNS_ROOT / "comparisons" / "sdim140-qubit-si1000-sd6"
comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_check = 1
styles = {
    "single_si1000": dict(color="#1f5fa8", marker="o", linestyle="-", label="one ancilla, SI1000"),
    "split_si1000": dict(color="#1f5fa8", marker="s", linestyle="--", label="two ancillas, SI1000"),
    "single_sd6": dict(color="#b5452b", marker="o", linestyle="-", label="one ancilla, SD6"),
    "split_sd6": dict(color="#b5452b", marker="s", linestyle="--", label="two ancillas, SD6"),
}
fig, ax = plt.subplots(figsize=(6.4, 4.4))
rows = []
for label, table_csv in table_csvs.items():
    frame = pd.read_csv(table_csv)
    frame = frame[(frame["check_num"] == comparison_check) & (frame["p"] > 0)].sort_values("p")
    lrb = frame[np.isfinite(frame["lrb_error_rate"]) & (frame["lrb_error_rate"] > 0)]
    yerr = np.vstack([
        lrb["lrb_error_rate"] - lrb["lrb_error_rate_ci_low"],
        lrb["lrb_error_rate_ci_high"] - lrb["lrb_error_rate"],
    ]).clip(min=0)
    ax.errorbar(lrb["p"], lrb["lrb_error_rate"], yerr=yerr, markersize=4, capsize=2, linewidth=1.2, **styles[label])
    rows.append(frame.assign(run=label))
for label, color in (("single_si1000", "#1f5fa8"), ("single_sd6", "#b5452b")):
    frame = pd.read_csv(table_csvs[label])
    frame = frame[(frame["check_num"] == comparison_check) & (frame["p"] > 0)].sort_values("p")
    rb = frame[np.isfinite(frame["rb_error_rate"]) & (frame["rb_error_rate"] > 0)]
    noise_title = "SI1000" if label.endswith("si1000") else "SD6"
    ax.plot(rb["p"], rb["rb_error_rate"], color=color, linestyle=":", marker="^", markersize=4,
            linewidth=1.2, label=f"physical RB, {noise_title}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(fit_config.p_axis_label)
ax.set_ylabel("Error rate per Clifford")
ax.set_title(r"$[[5,1,2]]_2$ LRB-D, check every Clifford, vs physical RB")
ax.grid(True, which="both", alpha=0.25)
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
comparison_pdf = comparison_dir / f"lrb-vs-rb-unif-{comparison_check}-four-configurations.pdf"
fig.savefig(comparison_pdf)
fig.savefig(comparison_pdf.with_suffix(".png"), dpi=160)
plt.show()
comparison_table = pd.concat(rows, ignore_index=True)[
    ["run", "p", "lrb_error_rate", "lrb_error_rate_ci_low", "lrb_error_rate_ci_high",
     "rb_error_rate", "rb_error_rate_ci_low", "rb_error_rate_ci_high"]
]
comparison_table.to_csv(comparison_dir / f"lrb-vs-rb-unif-{comparison_check}-four-configurations.csv", index=False)
print(comparison_pdf)

/pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/comparisons/sdim140-qubit-si1000-sd6/lrb-vs-rb-unif-1-four-configurations.pdf


### 9. GIF animations

The const=0 sweep over $p$ and the interval-check sweep at each $p$, per run. This is the slowest step, so it runs last; runs with a complete GIF set are reused.

In [10]:
# GIFs are the slowest step. A run whose GIF set (the const=0 sweep plus one
# interval sweep per p) is already complete is reused; the rest are rebuilt.
gif_outputs = {}
for label, plotter in plotters.items():
    animator = LRBGifAnimator(plotter, render_config=gif_render_config)
    const_gif = animator.out_dir / "const-0-rb-lrb-vs-p.gif"
    interval_gifs = sorted(animator.out_dir.glob("unif-interval-sweep-pidx*.gif"))
    if const_gif.exists() and len(interval_gifs) == len(plotter.probabilities):
        const_gif, interval_gifs = str(const_gif), [str(path) for path in interval_gifs]
    else:
        const_gif = animator.create_const_sweep_gif(const_check=0, strict=True)
        interval_gifs = animator.create_unif_interval_gifs_for_all_probabilities(
            interval_min=unif_check_min,
            interval_max=unif_check_max,
            strict=True,
        )
    gif_outputs[label] = (const_gif, interval_gifs)
    print(f"{label}: {Path(const_gif).name} + {len(interval_gifs)} interval GIFs in {animator.out_dir}")
display(Image(filename=gif_outputs["single_si1000"][0]))

single_si1000: const-0-rb-lrb-vs-p.gif + 15 interval GIFs in /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-55-folded_qubit-single-ancilla-si1000-sdim140/results/plots/gifs


single_sd6: const-0-rb-lrb-vs-p.gif + 15 interval GIFs in /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-03-folded_qubit-single-ancilla-sd6-sdim140/results/plots/gifs


split_si1000: const-0-rb-lrb-vs-p.gif + 15 interval GIFs in /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-12-58-folded_qubit_split_unif-split-ancilla-si1000-sdim140/results/plots/gifs


split_sd6: const-0-rb-lrb-vs-p.gif + 15 interval GIFs in /pscratch/sd/s/sohan100/worktrees/LRB-Simulations-sdim-2bc490701489f8e7/qubit-lrbd-sdim140/LRB-experiment-data-slurm/Run-2026-10-31-10-13-07-folded_qubit_split_unif-split-ancilla-sd6-sdim140/results/plots/gifs


<GIF animation, not stored; rerun the cell to show it>

## 10. Reading guide

Per-run PDFs, tables and GIFs are under each run's `results/plots/`; the four-configuration comparison is under `LRB-experiment-data-slurm/comparisons/sdim140-qubit-si1000-sd6/`. In every figure the horizontal axis is the two-qudit gate depolarizing probability $p$. SI1000 sets one-qudit gates to $p/10$, measurement flips to $5p$, reset flips and resonator idles to $2p$, and idles to $p/10$. SD6 sets every operation, idles included, to $p$. Physical RB at a given $p$ uses only one-qudit gates, so under SI1000 it sees $p/10$ per gate, and the LRB/RB pseudo-thresholds reflect that asymmetry.